# Atelier ML - Analyse d'image : utilisation du modèle

Objectif : utiliser le modèle entraîné dans `classification.ipynb` pour reconnaître des vêtements sur **de nouvelles images** trouvées sur Internet (dossier `images/`).

## Chargement du modèle

On recharge la description du réseau (`modele.json`) puis les poids appris pendant l'entraînement (`modele.weights.h5`).

In [1]:
#Chargement de la description du modèle
fichier_json = open('modele/modele.json', 'r')
modele_json = fichier_json.read()
fichier_json.close()

#Chargement de la description des poids du modèle
from keras.models import model_from_json
modele = model_from_json(modele_json)
#Chargement des poids
modele.load_weights("modele/modele.weights.h5")

## Catégories de classification

Dans le même ordre que les labels 0 à 9 du jeu de données.

In [2]:
#Définition des catégories de classifications
classes = ["Un T-shirt/haut",
           "Un pantalon",
           "Un pull", 
           "Une robe",
           "Un manteau",
           "Des sandales",
           "Une chemise",
           "Des baskets", 
           "Un sac",
           "Des bottes de cheville"]

## Préparation d'une image

Une image trouvée sur Internet doit être transformée pour ressembler aux images d'entraînement :

1. conversion en **niveaux de gris** (`convert('L')`)
2. **redimensionnement** à 20 pixels en gardant les proportions, puis centrage sur un carré blanc de 28 x 28
3. **inversion des couleurs et normalisation** : dans Fashion-MNIST, le vêtement est clair sur un fond noir, alors que les photos ont souvent un fond blanc. On inverse donc les pixels (255 - valeur) et on divise par 255 pour avoir des valeurs entre 0 et 1.

In [3]:
import numpy as np
from PIL import Image, ImageFilter

def load_and_preprocess_image(image_path, target_size=28, content_size=20):
    """
    Load an image, convert to grayscale, and preprocess for model input.

    Args:
        image_path: Path to the input image
        target_size: Size of the output square image (default: 28x28)
        content_size: Size of the resized content within the target (default: 20)

    Returns:
        numpy.ndarray: Preprocessed image array ready for model prediction
    """
    image = Image.open(image_path).convert('L')
    resized_img = resize_with_aspect_ratio(image, content_size)
    canvas = create_centered_image(resized_img, target_size)
    normalized_array = normalize_pixels(canvas)

    return normalized_array.reshape(1, target_size, target_size, 1)

def resize_with_aspect_ratio(image, target_dimension):
    """
    Resize image maintaining aspect ratio.

    Args:
        image: PIL Image object
        target_dimension: Target size for the longest dimension

    Returns:
        PIL Image: Resized and sharpened image
    """
    width, height = image.size

    if width > height:
        new_width = target_dimension
        new_height = max(1, round(target_dimension * height / width))
    else:
        new_height = target_dimension
        new_width = max(1, round(target_dimension * width / height))

    return image.resize((new_width, new_height), Image.LANCZOS).filter(ImageFilter.SHARPEN)

def create_centered_image(image, canvas_size):
    """
    Place image in the center of a white canvas.

    Args:
        image: PIL Image to center
        canvas_size: Size of the square canvas

    Returns:
        PIL Image: Image centered on white canvas
    """
    canvas = Image.new('L', (canvas_size, canvas_size), 255)

    img_width, img_height = image.size

    # Calculate centering offsets
    x_offset = (canvas_size - img_width) // 2
    y_offset = (canvas_size - img_height) // 2

    canvas.paste(image, (x_offset, y_offset))
    return canvas

def normalize_pixels(image):
    pixels = np.array(image.getdata(), dtype=np.float32)
    # Invert colors (white background to black) and normalize
    return (255 - pixels) / 255.0

## Prédiction

On soumet chaque image du dossier `images/` au modèle. Il retourne une probabilité pour chacune des 10 catégories, et `argmax` donne la catégorie la plus probable.

In [4]:
import os

for nom_image in sorted(os.listdir("images")):
    image_array = load_and_preprocess_image(os.path.join("images", nom_image))

    probabilities = modele.predict(image_array)[0]
    predicted_class = np.argmax(probabilities)

    print(f"===== {nom_image} =====")
    print(f"Selon moi l'image est : {classes[predicted_class]}\n")

    for i, class_name in enumerate(classes):
        print(f"{class_name}: {probabilities[i] * 100:.2f}%")
    print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
===== jean1.jpg =====
Selon moi l'image est : Un pantalon

Un T-shirt/haut: 11.12%
Un pantalon: 64.24%
Un pull: 8.61%
Une robe: 1.59%
Un manteau: 0.37%
Des sandales: 0.17%
Une chemise: 8.05%
Des baskets: 0.08%
Un sac: 5.77%
Des bottes de cheville: 0.00%

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
===== jean2.jpg =====
Selon moi l'image est : Un sac

Un T-shirt/haut: 0.66%
Un pantalon: 0.01%
Un pull: 0.05%
Une robe: 19.65%
Un manteau: 0.01%
Des sandales: 21.80%
Une chemise: 0.15%
Des baskets: 7.13%
Un sac: 50.48%
Des bottes de cheville: 0.06%

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
===== jean3.jpg =====
Selon moi l'image est : Des sandales

Un T-shirt/haut: 2.08%
Un pantalon: 0.17%
Un pull: 0.04%
Une robe: 9.33%
Un manteau: 0.43%
Des sandales: 51.77%
Une chemise: 0.15%
Des baskets: 28.36%
Un sac: 6.94%
Des bottes de cheville: 0.74%

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

C:\Users\Tommy1\AppData\Local\Temp\ipykernel_14852\2080474528.py:68: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = np.array(image.getdata(), dtype=np.float32)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
===== jean4.jpg =====
Selon moi l'image est : Un pantalon

Un T-shirt/haut: 33.84%
Un pantalon: 39.03%
Un pull: 0.14%
Une robe: 12.10%
Un manteau: 0.04%
Des sandales: 2.05%
Une chemise: 1.09%
Des baskets: 1.35%
Un sac: 10.35%
Des bottes de cheville: 0.00%



## Analyse des résultats

Les 4 images sont des jeans. Le modèle reconnaît **un pantalon** pour `jean1` et `jean4`, mais se trompe pour `jean2` (sac) et `jean3` (sandales).

Ces deux jeans sont **pliés**, vus de dos, en diagonale. Or les images de Fashion-MNIST montrent les pantalons **à plat, vus de face, à la verticale** : le modèle n'a jamais appris à quoi ressemble un jean plié. Une fois réduite à 28 x 28 pixels, la forme compacte d'un jean plié ressemble plus à un sac ou à une chaussure qu'à un pantalon.

Un modèle ne reconnaît bien que ce qui ressemble à ses données d'apprentissage. Pour reconnaître des vêtements pliés, il faudrait en ajouter dans le jeu d'apprentissage.